# Exploring experiment results

This notebook uses only the `moe_reliability_results` library, so it runs on any machine with access to a results directory:

```bash
uv sync --extra notebooks
uv run jupyter notebook notebooks/explore_results.ipynb
```

Set `RESULTS_DIR` below (or the `MOE_RESULTS_DIR` environment variable).

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

from moe_reliability_results import ResultsStore, plots, hta_plots

RESULTS_DIR = os.environ.get("MOE_RESULTS_DIR", "../results")
store = ResultsStore(RESULTS_DIR)
store

## Runs and dataset coverage

In [ ]:
store.configurations()[["run_id", "experiment", "status", "model_name", "n_npus", "batch_size", "n_points"]]

In [ ]:
configs = store.infrastructure_configurations()
print(f"{len(configs)} distinct infrastructure configurations with completed measurements")
configs.groupby("experiment").size()

## Summary table (one row per sweep point)

In [ ]:
summary = store.summary()
columns = ["run_id", "experiment", "model_name", "n_npus", "batch_size", "sweep_value", "point_status",
           "ttft_ms_mean", "tpot_ms_mean", "tpot_ms_p99", "trace_max_over_mean"]
summary[[c for c in columns if c in summary.columns]]

## Slow-down relative to the least imbalanced point of each run

For forced imbalance this is level 0 (the unmodified model); for synthetic workloads it is the smallest alpha.

In [ ]:
completed = summary[summary.point_status == "completed"].copy()
if "tpot_ms_mean" in completed:
    baseline = completed.sort_values("sweep_value").groupby("run_id")["tpot_ms_mean"].first()
    completed["tpot_slowdown"] = completed["tpot_ms_mean"] / completed["run_id"].map(baseline)
    display(completed.pivot_table(index=["experiment", "model_name", "n_npus", "batch_size"],
                                  columns="sweep_value", values="tpot_slowdown").round(2))

## Latency against the sweep value

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for experiment, ax in zip(["synthetic_workloads", "forced_imbalance"], axes):
    df = completed[completed.experiment == experiment]
    for (model, npus, batch), g in df.groupby(["model_name", "n_npus", "batch_size"]):
        g = g.sort_values("sweep_value")
        ax.plot(g.sweep_value, g.tpot_ms_mean, "o-", label=f"{model} npu{npus} bs{batch}")
    ax.set_title(experiment)
    ax.set_xlabel("alpha" if experiment == "synthetic_workloads" else "imbalance level")
    ax.set_ylabel("mean TPOT (ms)")
    if not df.empty:
        ax.legend(fontsize=8)
fig.tight_layout()

## Per-request distributions

In [ ]:
requests = store.requests()
if not requests.empty:
    display(requests.groupby(["experiment", "sweep_value"])[["ttft_ms", "tpot_ms"]].describe(percentiles=[0.5, 0.99]).round(1))

## Standard figures of a single run

In [ ]:
if len(store):
    run = store.latest()
    print(run)
    figures = plots.plot_run(run)
    list(figures)

## Synthetic workload quality

In [ ]:
synthetic = store.runs(experiment="synthetic_workloads")
if synthetic:
    run = synthetic[-1]
    for max_repeats in run.available_workloads():
        plots.plot_workload_sweep(run.workloads(max_repeats))

## Cross-run comparisons (profiled forced imbalance runs)

The overview is keyed by (model, batch size, imbalance level); restrict it to one accelerator count.

In [ ]:
forced = store.configurations(experiment="forced_imbalance")
if not forced.empty:
    K, T = plots.moe_imbalance_overview_inputs(store, n_npus=int(forced.n_npus.max()))
    if K and T:
        plots.plot_moe_imbalance_overview(K, T)
    print(f"kernel inputs: {len(K)}, latency inputs: {len(T)}")